In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [3]:
loader = TextLoader("backend/knowledge/german_law.md", encoding="utf-8")
docs = loader.load()

In [4]:
print(f"Loaded {len(docs)} document(s).")
print(f"Total characters in the file: {len(docs[0].page_content)}")

Loaded 1 document(s).
Total characters in the file: 8017


In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = text_splitter.split_documents(docs)

In [8]:
chunks = text_splitter.split_documents(docs)
# 'chunks' is now a list of many smaller Document objects.
# Each one has a slice of the original text.
print(f"\nSliced the document into {len(chunks)} chunks.")
print(f"\nExample - Chunk #1 content:\n---\n{chunks[0].page_content}\n---")
print(f"Example - Chunk #2 content:\n---\n{chunks[1].page_content}\n---")


Sliced the document into 23 chunks.

Example - Chunk #1 content:
---
# German Tenancy Law (Mietrecht) - Knowledge Base

This document serves as the core knowledge base for analyzing German residential lease agreements. It outlines key rights, illegal clauses, and relevant sections of the German Civil Code (BGB).

---
---
Example - Chunk #2 content:
---
## 1. Security Deposit (Kaution)
* **Legal Limit:** The security deposit may not exceed **three months' cold rent** (Kaltmiete). Any clause demanding more is invalid. (§ 551 Abs. 1 BGB)
* **Payment Terms:** The tenant has the right to pay the deposit in **three equal monthly installments**. The first installment is due at the start of the tenancy. (§ 551 Abs. 2 BGB)
---


In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [12]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vector_database = FAISS.from_documents(chunks, embeddings)

retriever = vector_database.as_retriever(search_kwargs={"k": 3})
print("Done!")

e:\Capstone Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\Capstone Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Wind\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate develope

Done!


In [13]:
test_results = retriever.invoke("Can my landlord keep my deposit?")
print(f"\nFAISS returned {len(test_results)} chunks for a test question:")
for i, doc in enumerate(test_results):
    print(f"\n--- Retrieved Chunk #{i+1} ---\n{doc.page_content}")


FAISS returned 3 chunks for a test question:

--- Retrieved Chunk #1 ---
* **Storage:** The landlord must keep the deposit separate from their personal assets, typically in a special escrow account, and it must accrue standard interest. (§ 551 Abs. 3 BGB)
* **Return:** There is no strict legal deadline for returning the deposit, but courts generally allow landlords 3 to 6 months to check for damages or outstanding utility bills.

--- Retrieved Chunk #2 ---
## 1. Security Deposit (Kaution)
* **Legal Limit:** The security deposit may not exceed **three months' cold rent** (Kaltmiete). Any clause demanding more is invalid. (§ 551 Abs. 1 BGB)
* **Payment Terms:** The tenant has the right to pay the deposit in **three equal monthly installments**. The first installment is due at the start of the tenancy. (§ 551 Abs. 2 BGB)

--- Retrieved Chunk #3 ---
* **Notice Required:** The landlord may only enter with a valid reason (e.g., maintenance, showing the apartment to prospective buyers/tenant

In [15]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()
llm = ChatOpenAI(
    openai_api_key=os.getenv("openai_api_key"),
    openai_api_base="https://openrouter.ai/api/v1",
    model_name="openrouter/auto-beta",
)

prompt = PromptTemplate.from_template("""
You are a German Tenancy Law expert. 
Answer the question using ONLY the following law excerpts as context.
If the context does not contain the answer, say: "I don't have enough legal information on this specific topic."
Law Context (retrieved from knowledge base):
{context}
User Question: {question}
Answer:
""")

rag_chain = (
    {"context":retriever,"question":RunnablePassthrough()}
    | prompt
    | llm 
    | StrOutputParser()
)

question = "Can I keep a small hamster in my apartment without asking my landlord?"
print(f"Question: {question}\n")
print("Processing through the RAG pipeline...")
response = rag_chain.invoke(question)
print(f"\n🛡️ MietShield Answer:\n{response}")

Question: Can I keep a small hamster in my apartment without asking my landlord?

Processing through the RAG pipeline...

🛡️ MietShield Answer:
Yes. Keeping small animals such as a hamster is always permitted and cannot be banned by the landlord.
